In [ ]:
%%bash
ncdump -h nc_full/782877.MLCAPE.sub.wrf2d_d01_2021-01-01_00:00:00.nc

In [1]:
import xarray as xr
import pandas as pd
import os
import glob

In [ ]:
# Load the NetCDF file
file_path = "nc_full/782877.MLCAPE.sub.wrf2d_d01_2021-01-01_00:00:00.nc"
ds = xr.open_dataset(file_path)

# Extract and flatten the MLCAPE values
mlcape_values = ds["MLCAPE"].values.flatten()

# Convert to a Pandas Series
mlcape_series = pd.Series(mlcape_values, name="MLCAPE")

# Display the first few values
print(mlcape_series.head())


 TSK Surface skin temperature, T2 temperature at 2 meters, SBCAPE Surface-based convective available potential energy (CAPE), MLCAPE Mixed-layer convective available potential energy (CAPE), and PWAT Precipitable water in the same from as we did above

In [ ]:
# Define the folder containing the .nc files
folder_path = "nc_full"

# Find all NetCDF files with "MLCAPE" in the filename
nc_files = glob.glob(os.path.join(folder_path, "*T2*.nc"))

# Initialize an empty list to store MLCAPE values
T2_list = []

# Process each NetCDF file
for file in nc_files:
    ds = xr.open_dataset(file)
    
    # Extract and flatten the MLCAPE values
    T2_values = ds["T2"].values.flatten()
    
    # Append to the list
    T2_list.extend(T2_values)
    
    ds.close()  # Close dataset to free memory

# Convert the list into a Pandas Series
T2_df = pd.DataFrame({"T2": T2_list})

# Display the first few rows
print(T2_df.head())


In [ ]:
# Define the output folder
df_folder = os.path.join(folder_path, "../df_full")

# Ensure the output folder exists
os.makedirs(df_folder, exist_ok=True)

# Define the output file path
output_file = os.path.join(df_folder, "T2_data.csv")

# Save DataFrame to CSV
T2_df.to_csv(output_file, index=False)

print(f"Saved DataFrame to {output_file}")

In [ ]:
# df = mlcape_df
# # Sort the damage values in ascending order
# df = df.sort_values(by="MLCAPE").reset_index(drop=True)

# #Set up Equation
# # Get the number of records
# n = len(df)
# # Compute empirical cumulative probabilities using Bernard's approximation
# df["F_hat"] = (df.index + 1 - 0.3) / (n + 0.4) #this is just the equation in python 
# # Find the damage value corresponding to the 95th percentile
# percentile_95 = df.loc[df["F_hat"] >= 0.95, "MLCAPE"].iloc[0] #filter F_hat to 95th and put into a variable

# print(f"95th percentile damage value: {percentile_95}") # text and variable printing

In [ ]:
import os
import glob
import xarray as xr
import pandas as pd

# Define the folder containing the .nc files
folder_path = "nc_full"

# Find all NetCDF files in the folder
nc_files = glob.glob(os.path.join(folder_path, "*.nc"))

# Initialize a set to store unique variable names
variables = set()

# First pass to collect variable names from the NetCDF files
for file in nc_files:
    ds = xr.open_dataset(file)
    variables.update(ds.data_vars.keys())  # Get all variable names
    ds.close()

# Now, loop through the list of variables
for var in variables:
    # Initialize an empty list to store the variable values
    var_list = []

    # Process each NetCDF file
    for file in nc_files:
        ds = xr.open_dataset(file)
        
        if var in ds.data_vars:  # Check if the variable exists in the dataset
            # Extract and flatten the variable values
            var_values = ds[var].values.flatten()
            
            # Append to the list
            var_list.extend(var_values)
        
        ds.close()  # Close dataset to free memory

    # Convert the list into a Pandas DataFrame
    var_df = pd.DataFrame({var: var_list})

    # Define the output folder for the DataFrames
    df_folder = os.path.join(folder_path, "../df_full")

    # Ensure the output folder exists
    os.makedirs(df_folder, exist_ok=True)

    # Define the output file path
    output_file = os.path.join(df_folder, f"{var}_data.csv")

    # Save DataFrame to CSV
    var_df.to_csv(output_file, index=False)

    print(f"Saved DataFrame for {var} to {output_file}")
